In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

from pathlib import Path
import joblib
from sklearn.isotonic import IsotonicRegression

Inputs

In [2]:
features_optimal_amount = pd.read_parquet(variables.FEATURES_OPTIMAL_AMOUNT_PATH)
features = pd.read_parquet(variables.FEATURES_DATASET_PATH)
targets  = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)

In [3]:
trained_models = []
for path in sorted(Path(variables.TRAINED_MODELS_PATH).glob("*.joblib")):
    filename = path.stem
    parts = filename.split("_", 1)
    horizon = int(parts[0][1:])
    model_name = parts[1]
    trained_models.append({"horizon": horizon, "model_name": model_name, "model": joblib.load(path)})

display(trained_models[:2])

[{'horizon': 1,
  'model_name': 'full_range_regressor_clipped_MAE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                learning_rate=0.025, max_bin=127, metric='mae',
                min_child_samples=25, n_estimators=1400, n_jobs=1, num_leaves=95,
                num_threads=1, objective='regression_l1', path_smooth=0.1,
                random_state=42, reg_alpha=0.05, reg_lambda=0.2, verbose=-1)},
 {'horizon': 1,
  'model_name': 'full_range_regressor_unclipped_RMSE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                learning_rate=0.025, max_bin=127, metric='rmse',
                min_child_samples=25, n_estimators=1200, n_jobs=1, num_leaves=95,
                num_threads=1, objective='regression', path_smooth=0.1,
                random_state=43, reg_alpha=0.05, reg_lambda=0.2, verbose=-1)}]

In [4]:
def build_model_dict(trained_models):
    horizons = sorted(set(m["horizon"] for m in trained_models))
    mm = {(m["horizon"], m["model_name"]): m["model"] for m in trained_models}
    return {
        "full_range_regressor_clipped_MAE_loss": [mm.get((h, "full_range_regressor_clipped_MAE_loss")) for h in horizons],
        "full_range_regressor_unclipped_RMSE_loss": [mm.get((h, "full_range_regressor_unclipped_RMSE_loss")) for h in horizons],
        "positive_spike_classifier_binary_loss": [mm.get((h, "positive_spike_classifier_binary_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_mae_loss": [mm.get((h, "positive_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_quantile_loss": [mm.get((h, "positive_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "negative_spike_classifer_unclipped_binary_loss": [mm.get((h, "negative_spike_classifer_unclipped_binary_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_mae_loss": [mm.get((h, "negative_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_quantile_loss": [mm.get((h, "negative_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "horizon_list":  horizons,
    }

model_dict = build_model_dict(trained_models)

pd.DataFrame(model_dict)[:3]

,full_range_regressor_clipped_MAE_loss,full_range_regressor_unclipped_RMSE_loss,positive_spike_classifier_binary_loss,positive_spike_regressor_unclipped_mae_loss,positive_spike_regressor_unclipped_quantile_loss,negative_spike_classifer_unclipped_binary_loss,negative_spike_regressor_unclipped_mae_loss,negative_spike_regressor_unclipped_quantile_loss,horizon_list
0,"LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.9, bagging_fraction=0.85...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.1, bagging_fraction=0.85...",1
1,"LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.9, bagging_fraction=0.85...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.1, bagging_fraction=0.85...",2
2,"LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.9, bagging_fraction=0.85...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.1, bagging_fraction=0.85...",3


In [5]:
X_validate = features[(features.index > variables.VALID_START) & (features.index <= variables.TEST_START)]
y_validate = targets[(targets.index > variables.VALID_START) & (targets.index <= variables.TEST_START)]

horizon_list = model_dict["horizon_list"]
full_range_regressor_clipped_MAE_loss = model_dict["full_range_regressor_clipped_MAE_loss"]
full_range_regressor_unclipped_RMSE_loss = model_dict["full_range_regressor_unclipped_RMSE_loss"]
positive_spike_classifier_binary_loss = model_dict["positive_spike_classifier_binary_loss"]
positive_spike_regressor_unclipped_mae_loss = model_dict["positive_spike_regressor_unclipped_mae_loss"]
positive_spike_regressor_unclipped_quantile_loss = model_dict["positive_spike_regressor_unclipped_quantile_loss"]
negative_spike_classifer_unclipped_binary_loss = model_dict["negative_spike_classifer_unclipped_binary_loss"]
negative_spike_regressor_unclipped_mae_loss = model_dict["negative_spike_regressor_unclipped_mae_loss"]

Helpers

In [ ]:
def get_features_for_this_horizon(h):
    feat_cols = [c for c in features_optimal_amount.loc[features_optimal_amount[f"h{h}"] == True, "feature"].tolist()]
    return X_validate.reindex(y_validate.index)[feat_cols].astype(np.float32)


def convert_from_asinh(y): return np.sinh(y) * variables.PRICE_TRANSFORM_SCALE

Core functions

In [ ]:
def Find_alpha_value_for_full_range_models():
    """
    The purpose of this functon is to find the best alpha (a mixing coeifficent that tells you how much weight
    to give each model's prediction when combining them) for each horizon

    Output: For each horizon, when combining these first two models, do so by this value per horizon.
    """

    horizon_full_range_blend_alphas = np.zeros(len(horizon_list), dtype=np.float32)

    # Iterate through the horizons
    for i, h in enumerate(horizon_list):
        
        # Get the features for this horizon
        X_validate_h = get_features_for_this_horizon(h)
        
        # Get the targets for this horizon
        y_validate_h = y_validate[f"target_h{h}"].values.astype(np.float32)

        # Use MODEL 1, to Make predictions (now on the validation data) for this horizon
        y_validate_h_pred_model_1 = convert_from_asinh(full_range_regressor_clipped_MAE_loss[i].predict(X_validate_h)).astype(np.float32)
        # Use MODEL 2, to Make predictions (now on the validation data) for this horizon
        y_validate_h_pred_model_2 = convert_from_asinh(full_range_regressor_unclipped_RMSE_loss[i].predict(X_validate_h)).astype(np.float32)

        
        # For this horizon, iterate again through a search space of alpha values
        best_alpha, best_mae = 0.0, float("inf")

        for alpha in np.linspace(0.0, 0.45, 10, dtype=np.float32):
            # Apply the alpha value to the model predictions
            y_validate_h_pred_model_1 = (1.0 - alpha) * y_validate_h_pred_model_1
            y_validate_h_pred_model_2 = alpha * y_validate_h_pred_model_2

            # Compute the prediction error with this alpha value
            error = y_validate_h - (y_validate_h_pred_model_1 + y_validate_h_pred_model_2)

            # Compute the MAE
            mae = float(np.mean(np.abs(error)))

            # Find the alpha that yeild the best MAE
            if mae < best_mae: 
                best_mae, best_alpha = mae, float(alpha)

        # Store the best alpha for each horizon
        horizon_full_range_blend_alphas[i] = best_alpha

    print(f"mean α={horizon_full_range_blend_alphas.mean():.3f}")
    print(f"min α={horizon_full_range_blend_alphas.min():.3f}")
    print(f"max α={horizon_full_range_blend_alphas.max():.3f}")
    display(horizon_full_range_blend_alphas[:5])
    return horizon_full_range_blend_alphas

horizon_full_range_blend_alphas = Find_alpha_value_for_full_range_models()

mean α=0.000
min α=0.000
max α=0.000


array([0., 0., 0., 0., 0.], dtype=float32)

In [ ]:
def model_combiner_full_range_blend(horizon_full_range_blend_alphas):
    """
    The purpose of this function is to blend the first two full range models using the prior calculated alphas
    """

    full_range_blend_by_horizon = {}

    # Iterate through the horizons
    for i, h in enumerate(horizon_list):
        
        # Get the features for this horizon
        X_validate_h = get_features_for_this_horizon(h)
        
        # Get the targets for this horizon
        y_validate_h = y_validate[f"target_h{h}"].values.astype(np.float32)

        # Create X_validate_h_naive, which is just a single feature, which is just same price last week
        X_validate_h_naive = X_validate_h.reindex(X_validate_h.index)[variables.NAIVE_BASELINE_PREDICTOR_COLUMN].values.astype(np.float32)
        
        # Use MODEL 1, to Make predictions (now on the validation data) for this horizon 
        y_validate_h_pred_model_1 = convert_from_asinh(full_range_regressor_clipped_MAE_loss[i].predict(X_validate_h)).astype(np.float32)
        
        # Use MODEL 2, to Make predictions (now on the validation data) for this horizon
        y_validate_h_pred_model_2 = convert_from_asinh(full_range_regressor_unclipped_RMSE_loss[i].predict(X_validate_h))

        alpha_small = (1.0 - horizon_full_range_blend_alphas[i])
        alpha_large = horizon_full_range_blend_alphas[i]

        # Actually apply the blend and merge the first two models predictions
        y_validate_h_pred_full_range_blend = (alpha_small * y_validate_h_pred_model_1 + alpha_large * y_validate_h_pred_model_2).astype(np.float32)
        full_range_blend_by_horizon[h] = y_validate_h_pred_full_range_blend

    return full_range_blend_by_horizon

full_range_blend_by_horizon = model_combiner_full_range_blend(horizon_full_range_blend_alphas)

In [ ]:
def model_combiner(
        spike_adjustment_mode, 
        spike_source_mode,
        PROBABILITY_SHRESHHOLD, 
        PROBABILITY_POWER, 
        MAX_UPLIFT_WEIGHT
    ):

    """
    The purpose of this function is to combine full-range and spike-model predictions using configurable spike adjustment rules.
    """
  
    # Wether to use the P90 spike, regressor, or largest of the two
    if spike_source_mode == 0: 
        spike_source = y_validate_h_pred_model_5

    elif spike_source_mode == 1: 
        spike_source = np.maximum(y_validate_h_pred_model_4, y_validate_h_pred_model_5).astype(np.float32)

    elif spike_source_mode == 2:
        spike_source = y_validate_h_pred_model_4
    
    # 0: probability-weighted mix between base prediction and spike source.
    if spike_adjustment_mode == 0: 
        probability_of_no_spike = (1.0 - y_validate_h_pred_model_3)
        probability_weighted_spike_value = y_validate_h_pred_model_3 * spike_source
        y_validate_h_pred_spike_blend = (probability_of_no_spike * y_validate_h_pred_full_range_blend + probability_weighted_spike_value).astype(np.float32)
        
        return y_validate_h_pred_spike_blend
    
    # 1: smooth nonlinear uplift based on spike probability.
    elif spike_adjustment_mode == 1: 
        # This is “how far above threshold we are”.
        spike_probability_above_threshold = y_validate_h_pred_model_3 - PROBABILITY_SHRESHHOLD
        # This is the maximum possible distance above threshold (from threshold up to 1.0). +1e-6 prevents divide-by-zero if threshold is set to 1.0.
        max_spike_probability_above_threshold  = 1.0 - PROBABILITY_SHRESHHOLD + 1e-6
        # Converts that distance into a normalized ratio in [0, 1].
        normalized_spike_probability_above_threshold = np.clip((spike_probability_above_threshold) / (max_spike_probability_above_threshold), 0.0, 1.0)
        
        # Spike confidence shaping term before max cap.
        shaped_spike_confidence = np.power(normalized_spike_probability_above_threshold, PROBABILITY_POWER)
        # Final uplift weight applied to the positive spike gap.
        shaped_spike_confidence_weighted = (shaped_spike_confidence * MAX_UPLIFT_WEIGHT).astype(np.float32)
        # Only the upward gap between base prediction and spike source.
        positive_spike_gap = np.maximum(spike_source - y_validate_h_pred_full_range_blend, 0.0)
        # Actual uplift amount added to base prediction.
        weighted_spike_uplift = positive_spike_gap * shaped_spike_confidence_weighted

        y_validate_h_pred_spike_blend = (y_validate_h_pred_full_range_blend + weighted_spike_uplift).astype(np.float32)
        
        return y_validate_h_pred_spike_blend
    
    # 2: hard gate uplift (only adjust when probability passes threshold).
    elif spike_adjustment_mode == 2: 
        # boolean mask for rows where spike probability is high enough.
        spike_probability_above_threshold_gate = y_validate_h_pred_model_3 >= PROBABILITY_SHRESHHOLD
        y_validate_h_pred_full_range_blend = y_validate_h_pred_full_range_blend.copy()

        # Base prediction values at gated rows
        base_predictions_at_gated_rows = y_validate_h_pred_full_range_blend[spike_probability_above_threshold_gate]
        # Those base values shifted by PROBABILITY_POWER
        shifted_base_at_gated_rows = base_predictions_at_gated_rows + float(PROBABILITY_POWER)
        # Spike-source gap at gated rows.
        spike_source_gap_at_gated_rows = spike_source[spike_probability_above_threshold_gate] - y_validate_h_pred_full_range_blend[spike_probability_above_threshold_gate]
        # Positive_spike_gap_at_gated_rows
        positive_spike_gap_at_gated_rows = np.maximum(spike_source_gap_at_gated_rows, 0.0)
        
        y_validate_h_pred_full_range_blend[spike_probability_above_threshold_gate] = shifted_base_at_gated_rows * positive_spike_gap_at_gated_rows

        y_validate_h_pred_spike_blend = y_validate_h_pred_full_range_blend.astype(np.float32)

        return y_validate_h_pred_spike_blend


# Iterate through the horizons
for i, h in enumerate(horizon_list):

    X_validate_h = get_features_for_this_horizon(h)

    y_validate_h_pred_full_range_blend = full_range_blend_by_horizon[h]

    # Use MODEL 3, to Make predictions (as a confidence %, instead of just spike or no spike) (now on the validation data) for this horizon
    y_validate_h_pred_model_3 = positive_spike_classifier_binary_loss[i].predict_proba(X_validate_h)[:, 1].astype(np.float32)
    # Use MODEL 4, to Make predictions (now on the validation data) for this horizon
    y_validate_h_pred_model_4 = convert_from_asinh(positive_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h)).astype(np.float32) 
    # Use MODEL 5, to Make predictions (now on the validation data) for this horizon
    y_validate_h_pred_model_5 = convert_from_asinh(positive_spike_regressor_unclipped_quantile_loss[i].predict(X_validate_h)).astype(np.float32) 

    y_validate_h_pred_spike_blend = model_combiner(
        spike_adjustment_mode=0,
        spike_source_mode=0,
        PROBABILITY_SHRESHHOLD=0.0,
        PROBABILITY_POWER=0.0,
        MAX_UPLIFT_WEIGHT=0.0
    )

In [ ]:
def get_best_model_combiner_params():
    
    # Tuning parameters
    spike_adjustment_modes = np.zeros(len(horizon_list), dtype=np.int8)
    spike_source_modes = np.zeros(len(horizon_list), dtype=np.int8)
    probability_threshholds = np.full(len(horizon_list), 0.20, dtype=np.float32)
    probability_powers = np.full(len(horizon_list), 2.0,  dtype=np.float32)
    max_uplift_powers = np.full(len(horizon_list), 0.75, dtype=np.float32)
        
    # Iterate through the horizons
    for i, h in enumerate(horizon_list):
        
        X_validate_h = get_features_for_this_horizon(h)
        y_validate_h_pred_full_range_blend = full_range_blend_by_horizon[h]
        y_validate_h_pred_model_3 = positive_spike_classifier_binary_loss[i].predict_proba(X_validate_h)[:, 1].astype(np.float32)
        y_validate_h_pred_model_4 = convert_from_asinh(positive_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h)).astype(np.float32)
        y_validate_h_pred_model_5 = convert_from_asinh(positive_spike_regressor_unclipped_quantile_loss[i].predict(X_validate_h)).astype(np.float32)
        
        # Get the targets for this horizon
        y_validate_h = y_validate[f"target_h{h}"].values.astype(np.float32)

        spike_flag = y_validate_h > variables.SPIKE_THRESHOLD
        dip_flag = y_validate_h < variables.DIP_THRESHOLD

        # Helper function to calculate MAE between actual prices and predictions
        def MAE_error_calculator(y_validate_h_pred_spike_blend, filter=None):
            if filter is None:
                filter = np.ones_like(y_validate_h, dtype=bool)
            if int((filter).sum()) > 0:
                error = y_validate_h[filter] - y_validate_h_pred_spike_blend[filter]
            else:
                print("no events of this kind")
                return float("inf")
            error = np.abs(error)
            error = np.mean(error)
            error = float(error)

            return error

        # Calculate inital MAE for benchmark
        y_validate_h_pred_spike_blend = model_combiner(0,0,0.0,0.0,0.0)
        inital_non_spike_MAE = MAE_error_calculator(y_validate_h_pred_spike_blend,~spike_flag)
     
        def evaluate_each_model_combiner_paramters(y_validate_h_pred_spike_blend):

            # Check the error of each model
            spike_MAE = MAE_error_calculator(y_validate_h_pred_spike_blend, spike_flag)
            non_spike_MAE = MAE_error_calculator(y_validate_h_pred_spike_blend, ~spike_flag)
            dip_MAE = MAE_error_calculator(y_validate_h_pred_spike_blend, dip_flag)
            all_MAE = MAE_error_calculator(y_validate_h_pred_spike_blend)
            
            # if non spike error increases by 20% that give this call the worst possible score
            if non_spike_MAE > inital_non_spike_MAE + 0.20:
                return float("inf")
            
            # Return an overall error based on the parameters
            # Increase the overall error by panalising certain components
            error = 2.2 * spike_MAE + 1.3 * dip_MAE + 1.0 * non_spike_MAE + 0.25 * all_MAE
            
            return error
          
        # These are the 5 per horizon paramaters + the overall error of each 5
        best_error = evaluate_each_model_combiner_paramters(model_combiner(0,0,0.0,0.0,0.0))
        best_spike_adjustment_mode = 0
        best_spike_source_mode = 0
        best_PROBABILITY_SHRESHHOLD = 0.0
        best_PROBABILITY_POWER = 0.0
        best_MAX_UPLIFT_WEIGHT = 0.0

        # Go through a bunch of combinations of the parameters
        PROBABILITY_THRESHOLD_GRID = np.array([0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50], dtype=np.float32)
        PROBABILITY_POWER_GRID = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0], dtype=np.float32)
        MAX_UPLIFT_WEIGHT_GRID = np.array([0.40, 0.60, 0.80, 1.00, 1.20], dtype=np.float32)

        for spike_adjustment_mode in (0, 1, 2):
            for spike_source_mode in (0, 1, 2):
                for PROBABILITY_SHRESHHOLD in PROBABILITY_THRESHOLD_GRID:
                    for PROBABILITY_POWER in PROBABILITY_POWER_GRID:
                        for MAX_UPLIFT_WEIGHT in MAX_UPLIFT_WEIGHT_GRID:
                            
                            # combine the models with these param and return error
                            error = evaluate_each_model_combiner_paramters(
                                model_combiner(
                                    spike_adjustment_mode,
                                    spike_source_mode,
                                    PROBABILITY_SHRESHHOLD,
                                    PROBABILITY_POWER,
                                    MAX_UPLIFT_WEIGHT
                                    )
                                )

                            # if params combination yeilded better result, save these params
                            if error < best_error: 
                                best_error = error
                                best_spike_adjustment_mode = spike_adjustment_mode
                                best_spike_source_mode = spike_source_mode
                                best_PROBABILITY_SHRESHHOLD = PROBABILITY_SHRESHHOLD
                                best_PROBABILITY_POWER = PROBABILITY_POWER
                                best_MAX_UPLIFT_WEIGHT = MAX_UPLIFT_WEIGHT
                
        spike_adjustment_modes[i] = best_spike_adjustment_mode
        spike_source_modes[i] = best_spike_source_mode
        probability_threshholds[i] = best_PROBABILITY_SHRESHHOLD
        probability_powers[i] = best_PROBABILITY_POWER
        max_uplift_powers[i] = best_MAX_UPLIFT_WEIGHT

    return spike_adjustment_modes, spike_source_modes, probability_threshholds, probability_powers, max_uplift_powers

In [ ]:
def tune_dip_blending():  

    # dip_kind = np.zeros(len(horizon_list), dtype=np.int8)
    # dip_p1 = np.full(len(horizon_list), 0.15, dtype=np.float32)
    # dip_p2 = np.full(len(horizon_list), 1.0,  dtype=np.float32)
    # dip_p3 = np.full(len(horizon_list), 0.60, dtype=np.float32)

    # def _apply_dip_policy(y_base, dip_prob, y_dip, kind, p1, p2, p3):
#     if kind == 0: return ((1.0 - dip_prob) * y_base + dip_prob * y_dip).astype(np.float32)

#     def _dip_blend(y_base, dip_prob, y_dip, p_thr, p_pow, w_max):
#         p_adj = np.clip((dip_prob - p_thr) / (1.0 - p_thr + 1e-6), 0.0, 1.0)
#         w = (np.power(p_adj, p_pow) * w_max).astype(np.float32)
#         return (y_base - w * np.maximum(y_base - y_dip, 0.0)).astype(np.float32)

#     if kind == 1: return _dip_blend(y_base, dip_prob, y_dip, p1, p2, p3)
#     gate = dip_prob >= p1
#     out = y_base.copy()
#     out[gate] = y_base[gate] - float(p2) * np.maximum(y_base[gate] - y_dip[gate], 0.0)
#     return out.astype(np.float32)

        DIP_THRESHOLD_GRID = np.array([0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30], dtype=np.float32)
        DIP_POW_GRID = np.array([0.5, 1.0, 1.5, 2.0, 3.0], dtype=np.float32)
        DIP_WMAX_GRID = np.array([0.30, 0.50, 0.70, 0.90], dtype=np.float32)
        DIP_GATE_W_GRID = np.array([0.40, 0.60, 0.80, 1.00], dtype=np.float32)

        dp = negative_spike_classifer_unclipped_binary_loss[i].predict_proba(X_validate_h)[:, 1].astype(np.float32)
        dr = convert_from_asinh(negative_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h)).astype(np.float32)
        y_tune = y_validate_h[X_validate_h_naive]; y_validate_h_pred_full_range_blend = y_model[X_validate_h_naive]; p_tune = dp[X_validate_h_naive]; d_tune = dr[X_validate_h_naive]
        ref_mid = (y_tune >= variables.DIP_THRESHOLD) & (y_tune <= variables.SPIKE_THRESHOLD)
        ref_mid_mae = float(np.mean(np.abs(y_tune[ref_mid] - y_validate_h_pred_full_range_blend[ref_mid]))) if int(ref_mid.sum()) > 0 else float(np.mean(np.abs(y_tune - y_validate_h_pred_full_range_blend)))
        best_d_sc = _spike_policy_score(y_tune, y_validate_h_pred_full_range_blend, variables.SPIKE_THRESHOLD)
        best_dk, best_dp1, best_dp2, best_dp3 = 0, 0.0, 0.0, 0.0

        def _accept_dip(yc):
            mm = float(np.mean(np.abs(y_tune[ref_mid] - yc[ref_mid]))) if int(ref_mid.sum()) > 0 else float(np.mean(np.abs(y_tune - yc)))
            if mm > ref_mid_mae + 0.15: return False, float("inf")
            return True, _spike_policy_score(y_tune, yc, variables.SPIKE_THRESHOLD)

        ok, sc = _accept_dip(_apply_dip_policy(y_validate_h_pred_full_range_blend, p_tune, d_tune, 0, 0.0, 0.0, 0.0))
        if ok and sc < best_d_sc: best_d_sc = sc; best_dk = 0; best_dp1, best_dp2, best_dp3 = 0.0, 0.0, 0.0
        for thr in DIP_THRESHOLD_GRID:
            for pwr in DIP_POW_GRID:
                for wmx in DIP_WMAX_GRID:
                    ok, sc = _accept_dip(_apply_dip_policy(y_validate_h_pred_full_range_blend, p_tune, d_tune, 1, float(thr), float(pwr), float(wmx)))
                    if ok and sc < best_d_sc: best_d_sc = sc; best_dk = 1; best_dp1, best_dp2, best_dp3 = float(thr), float(pwr), float(wmx)
        for thr in DIP_THRESHOLD_GRID:
            for gw in DIP_GATE_W_GRID:
                ok, sc = _accept_dip(_apply_dip_policy(y_validate_h_pred_full_range_blend, p_tune, d_tune, 2, float(thr), float(gw), 0.0))
                if ok and sc < best_d_sc: best_d_sc = sc; best_dk = 2; best_dp1, best_dp2, best_dp3 = float(thr), float(gw), 0.0

        dip_kind[i] = best_dk; dip_p1[i] = best_dp1; dip_p2[i] = best_dp2; dip_p3[i] = best_dp3
        y_model = _apply_dip_policy(y_model, dp, dr, int(best_dk), float(best_dp1), float(best_dp2), float(best_dp3))

        # Other
        y_t = y_validate_h[X_validate_h_naive]; y_m = y_model[X_validate_h_naive]; y_n = X_validate_h_naive[X_validate_h_naive]
        best_a, best_mae_v = 1.0, float("inf")
        for a in np.linspace(0.0, 1.0, 41, dtype=np.float32):
            mae = float(np.mean(np.abs(y_t - (a * y_m + (1.0 - a) * y_n))))
            if mae < best_mae_v: best_mae_v, best_a = mae, float(a)
        blend_alphas[i] = best_a


Step C)

In [9]:
def C():
    # Step C: Isotonic calibration
    calibrators = [None] * len(horizon_list)
    for i, h in enumerate(horizon_list):
    
        X_validate_h = get_features_for_this_horizon(h)
        y_l1 = convert_from_asinh(full_range_regressor_clipped_MAE_loss[i].predict(X_validate_h)).astype(np.float32)
        y_base_val = y_l1
        if full_range_regressor_unclipped_RMSE_loss[i] is not None:
            y_base_val = ((1.0 - blend_l2_alphas[i]) * y_l1 + blend_l2_alphas[i] * convert_from_asinh(full_range_regressor_unclipped_RMSE_loss[i].predict(X_validate_h))).astype(np.float32)
        y_model = y_base_val
        if positive_spike_classifier_binary_loss[i] is not None:
            sp = positive_spike_classifier_binary_loss[i].predict_proba(X_validate_h)[:, 1].astype(np.float32)
            sr = convert_from_asinh(positive_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h)).astype(np.float32) if positive_spike_regressor_unclipped_mae_loss[i] else y_base_val
            sq = convert_from_asinh(positive_spike_regressor_unclipped_quantile_loss[i].predict(X_validate_h)).astype(np.float32) if positive_spike_regressor_unclipped_quantile_loss[i] else sr
            y_model = _apply_spike_policy(y_base_val, sp, sr, sq, int(spike_kind[i]), float(spike_p1[i]), float(spike_p2[i]), float(spike_p3[i]), int(spike_src[i]))
        if negative_spike_classifer_unclipped_binary_loss[i] is not None and negative_spike_regressor_unclipped_mae_loss[i] is not None:
            dp = negative_spike_classifer_unclipped_binary_loss[i].predict_proba(X_validate_h)[:, 1].astype(np.float32)
            dr = convert_from_asinh(negative_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h)).astype(np.float32)
            y_model = _apply_dip_policy(y_model, dp, dr, int(dip_kind[i]), float(dip_p1[i]), float(dip_p2[i]), float(dip_p3[i]))
        y_naive = X_validate.reindex(val_tgt.index)[f"{variables.SELECTED_TARGET_COLUMN_NAME}_lag_336"].values.astype(np.float32)
        y_blend = float(blend_alphas[i]) * y_model + (1.0 - float(blend_alphas[i])) * y_naive
        y_true = val_tgt[f"target_h{h}"].values.astype(np.float32)
        mask = np.isfinite(y_true) & np.isfinite(y_blend)
        if int(mask.sum()) < 500: continue
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(y_blend[mask], y_true[mask])
        calibrators[i] = iso

Export

In [10]:
def export():
    model_dict.update(dict(
        blend_l2_alphas = blend_l2_alphas,
        blend_alphas = blend_alphas,

        spike_kind = spike_kind,
        spike_src = spike_src,
        spike_p1 = spike_p1,
        spike_p2 = spike_p2,
        spike_p3 = spike_p3,

        dip_kind = dip_kind,
        dip_p1 = dip_p1,
        dip_p2 = dip_p2,
        dip_p3 = dip_p3,

        calibrators=calibrators,
    ))
    return model_dict
